# Stage 0 — Data Loading & Exploration
Yelp Open Dataset → cleaned DataFrame with sentiment labels and silver tag labels.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from preprocessing import (
    load_yelp_reviews, preprocess_dataframe,
    get_tag_columns, train_val_test_split,
    build_restaurant_profiles, make_sample_dataset,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# ----------------------------------------------------------------
# Option A: real Yelp data (update paths after downloading from Kaggle)
# ----------------------------------------------------------------
REVIEWS_PATH   = '../data/yelp_academic_dataset_review.json'
BUSINESSES_PATH = '../data/yelp_academic_dataset_business.json'
USE_SAMPLE = not os.path.exists(REVIEWS_PATH)

if USE_SAMPLE:
    print('Yelp files not found — using synthetic sample dataset (2000 rows).')
    raw_df = make_sample_dataset(n=2000)
else:
    raw_df = load_yelp_reviews(REVIEWS_PATH, BUSINESSES_PATH, max_reviews=500_000)

print(raw_df.shape)
raw_df.head(3)

In [ ]:
df = preprocess_dataframe(raw_df)
df.to_parquet('../data/processed.parquet', index=False)
print('Saved to data/processed.parquet')
df.head(3)

In [ ]:
# Sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#f1c40f','#2ecc71'])
axes[0].set_title('Sentiment Label Distribution')
axes[0].set_xlabel('')

df['stars'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Star Rating Distribution')
axes[1].set_xlabel('Stars')
plt.tight_layout()
plt.savefig('../data/eda_sentiment_stars.png', dpi=150)
plt.show()

In [ ]:
# Tag frequency
tag_cols = get_tag_columns(df)
tag_freq = df[tag_cols].sum().sort_values(ascending=True)
tag_freq.index = [c.replace('tag_', '').replace('_', ' ').title() for c in tag_freq.index]

fig, ax = plt.subplots(figsize=(8, 5))
tag_freq.plot(kind='barh', ax=ax, color='coral')
ax.set_title('Tag Frequency (heuristic silver labels)')
ax.set_xlabel('# Reviews')
plt.tight_layout()
plt.savefig('../data/eda_tag_frequency.png', dpi=150)
plt.show()

In [ ]:
# Review length distribution
df['text_len'] = df['text_clean'].str.split().str.len()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['text_len'].clip(upper=300), bins=50, color='steelblue', edgecolor='white')
ax.set_xlabel('Token count')
ax.set_title('Review Length Distribution (clipped at 300 tokens)')
plt.tight_layout()
plt.savefig('../data/eda_review_length.png', dpi=150)
plt.show()

In [ ]:
# Train / val / test split
train, val, test = train_val_test_split(df)
print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')

# Persist splits
train.to_parquet('../data/train.parquet', index=False)
val.to_parquet('../data/val.parquet', index=False)
test.to_parquet('../data/test.parquet', index=False)
print('Splits saved.')

In [ ]:
# Restaurant profiles
profiles = build_restaurant_profiles(df)
profiles.to_parquet('../data/restaurant_profiles.parquet', index=False)
print(f'{len(profiles):,} restaurant profiles saved.')
profiles.head()